In [ ]:
%pip install -q -r ../requirements-notebooks.txt

# 01 — Explore river data

Goal: load all CSVs, clean glitches, resample to 1-min grid, then characterize the upstream → Centro signal via cross-correlation. Output of this notebook feeds Phase 2 (SARIMAX lags).

Napo (station-02) and Balinad (station-03) are upstream of Centro (station-01). We expect their water level / flow rate to lead Centro's by some lag — that's the actual ML signal.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf, adfuller, pacf

from lib.preprocess import build_wide, cross_correlation, load_csv_dir

DATASET_DIR = Path.cwd().parent.parent / "dataset"
DATASET_DIR

In [ ]:
long = load_csv_dir(DATASET_DIR)
print(long.shape, long['station_id'].value_counts().to_dict())
long.head()

In [ ]:
wide = build_wide(long)
print(f'Wide shape: {wide.shape}')
print(f'Time range: {wide.index.min()} -> {wide.index.max()}')
wide.describe()

## Time-series visualization (post-cleaning)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for sid, color in [('station-01', '#4fc3f7'), ('station-02', '#81c784'), ('station-03', '#ffb74d')]:
    axes[0].plot(wide.index, wide[f'{sid}_water_level'], label=sid, color=color, lw=0.8)
    axes[1].plot(wide.index, wide[f'{sid}_flow_rate'], label=sid, color=color, lw=0.8)
axes[0].set_ylabel('Water level (cm)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_ylabel('Flow rate (L/min)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()

## Cross-correlation: upstream stations vs Centro

Positive lag = upstream series leads Centro (the expected direction).

In [ ]:
centro = wide['station-01_water_level']
results = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
pairs = [
    ('station-02_water_level', 'Napo level'),
    ('station-02_flow_rate', 'Napo flow'),
    ('station-03_water_level', 'Balinad level'),
    ('station-03_flow_rate', 'Balinad flow'),
]
for ax, (col, label) in zip(axes.flat, pairs):
    lags, corr = cross_correlation(centro, wide[col], max_lag_min=60)
    ax.bar(lags, corr, width=1, color='#4fc3f7')
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'corr(Centro level, {label}) — peak at lag={lags[np.nanargmax(np.abs(corr))]} min, r={np.nanmax(np.abs(corr)):.2f}')
    ax.set_xlabel('lag (min, positive = upstream leads)')
    ax.grid(alpha=0.3)
    results[col] = (int(lags[np.nanargmax(np.abs(corr))]), float(np.nanmax(np.abs(corr))))
plt.tight_layout()
results

**Interpretation:** if peak \|r\| < 0.3 on this calm dataset, the upstream signal is not yet visible — expected, since we haven't captured a rain event. The lag values still serve as priors for Phase 2 SARIMAX. Re-run after the first storm to validate.

## Stationarity + ACF/PACF for Centro level (informs SARIMAX order)

In [ ]:
centro_clean = centro.dropna()
adf = adfuller(centro_clean)
print(f'ADF stat={adf[0]:.3f}, p-value={adf[1]:.4f}  ->  {"stationary" if adf[1] < 0.05 else "NON-stationary, will need d=1"}')

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].stem(acf(centro_clean, nlags=60))
axes[0].set_title('ACF (60 min)'); axes[0].grid(alpha=0.3)
axes[1].stem(pacf(centro_clean, nlags=60))
axes[1].set_title('PACF (60 min)'); axes[1].grid(alpha=0.3)
plt.tight_layout()

**Save lag results for Phase 2:**

In [ ]:
import json
out_path = Path.cwd().parent / 'lags.json'
json.dump({k: {'lag_min': v[0], 'abs_corr': v[1]} for k, v in results.items()}, out_path.open('w'), indent=2)
print(f'Wrote {out_path}')